In [ ]:
%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from loguru import logger
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.generation.graphrag import GraphRAG
from neo4j_graphrag.generation.prompts import PromptTemplate
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.retrievers import HybridRetriever
from neo4j_graphrag.types import LLMMessage
from openai import OpenAI

from core import chat
from core.models import Settings

VECTOR_INDEX = "coins_embeddings_index"
FULL_TEXT_INDEX = "coins_text_index"
settings = Settings()


def find_coins_by_imperor(name: str) -> list[str]:
    driver = GraphDatabase.driver(
        settings.neo4j_uri,
        auth=(settings.neo4j_username, settings.neo4j_password),
    )
    with driver.session() as session:
        result = session.run(
            """
            MATCH (c:Coin)-[:HAS_AUTHORITY]->(e:Authority {name: $name})
            RETURN c LIMIT 10
            """,
            name=name,
        )

        data = [{
            "record_id": record["c"]["record_id"],
            "title": record["c"]["title"],
            "date_range": record["c"]["date_range"],
            "from_year": record["c"]["from_year"],
            "to_year": record["c"]["to_year"],
            "object_type": record["c"]["object_type"],
            "description": record["c"]["description"],
        } for record in result]
    driver.close()
    return data


coins = find_coins_by_imperor("Caracalla")
coins